In [1]:
import os
import sys
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

from train.helpers import gen_clarify_q_prompt

/home/gosh/Desktop/Work/AUA/sft_demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
base_model_id = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.unk_token or tokenizer.eos_token
tokenizer.padding_side = "left"

In [3]:
model = AutoModelForCausalLM.from_pretrained(base_model_id, device_map=DEVICE)

adapter_path = os.path.join(PROJECT_ROOT, "aua/meta-llama/Llama-3.2-3B/gen_clarify_q/Llama-3.2-3B/best_checkpoint")
model = PeftModel.from_pretrained(model, adapter_path, is_trainable=False)

Loading weights: 100%|██████████| 254/254 [00:00<00:00, 398.18it/s]


In [12]:
prompt = gen_clarify_q_prompt(qa_input="What is my name?")
print(prompt)

Question: What is my name?
Clarifying Question:


In [13]:
encoded = tokenizer(prompt, return_tensors="pt")
input_ids = encoded.input_ids.to(DEVICE)
attention_mask = encoded.attention_mask.to(DEVICE)

answer = model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_new_tokens=100,
    pad_token_id=tokenizer.pad_token_id,
)

In [14]:
new_tokens = answer[0][input_ids.shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

 Are you asking about your name in English, French, or Spanish?

